Este projeto implementa um assistente de IA generativa utilizando Python e LLMs, com foco em construção de pipelines inteligentes e integração com dados externos. São aplicados conceitos de engenharia de prompts, uso de PromptTemplate/ChatPromptTemplate, geração de saídas estruturadas (JSON/stream) e técnicas de Retrieval-Augmented Generation (RAG) para consulta de documentos (.txt e .pdf).

Como caso de uso, foi desenvolvido um assistente para planejamento de viagens capaz de sugerir locais, analisar informações contextuais e responder de forma orientada a dados. A solução prioriza modularidade, escalabilidade e boas práticas no desenvolvimento de aplicações baseadas em IA

 LangChain = É um framework que ajuda a construir aplicações com IA generativa
 
 LLM = Modelo de linguagem usado é da OpenAI (Token)

In [0]:
### requirements.txt
%pip install \
  openai==1.86.0 \
  langchain==0.3.25 \
  langchain-openai==0.3.22 \
  langgraph==0.4.8 \
  python-dotenv==1.1.0 \
  faiss-cpu==1.11.0 \
  langchain-community==0.3.25 \
  pypdf==5.6.0


Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
%pip install langchain-community

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()

In [0]:
OPENAI_API_KEY = "teste"

In [0]:
from langchain_openai import ChatOpenAI
from langchain.prompts import PromptTemplate
import os 

api_key=OPENAI_API_KEY

numero_dias= 7
numero_criancas= 2
atividade= "natureza"

modelo_de_prompt = PromptTemplate(
   template = """
   Criar um roteiro de viagem de {numero_dias} dias,
    para uma familiar com {numero_criancas} criancas,
    que gostam de {atividade}
   """
)

prompt = modelo_de_prompt.format(
    numero_dias=numero_dias,
    numero_criancas=numero_criancas,
    atividade=atividade
)
print( "Prompt: \n", prompt)

modelo = ChatOpenAI(
    model="gpt-3.5-turbo",
    temperature=0.5,  #### nível de criatividade do modelo: quanto mais próximo de 2, mais criativo; quanto mais próximo de 0, menos criativo
    api_key=api_key
)
resposta = modelo.invoke(prompt)
print(resposta.content)

Prompt: 
 
   Criar um roteiro de viagem de 7 dias,
    para uma familiar com 2 criancas,
    que gostam de natureza
   
Dia 1: Chegada na cidade de Bonito, Mato Grosso do Sul
- Check-in em hotel/pousada
- Visita ao Balneário Municipal para um dia de diversão na água
- Jantar em um restaurante local

Dia 2: Gruta do Lago Azul e Aquário Natural
- Passeio pela manhã na Gruta do Lago Azul para apreciar a beleza das formações rochosas e do lago
- Almoço em um restaurante próximo
- Tarde no Aquário Natural, onde as crianças podem nadar e observar a vida marinha

Dia 3: Flutuação no Rio da Prata
- Dia de aventura no Rio da Prata, com flutuação em águas cristalinas e observação da fauna e flora local
- Almoço em um restaurante típico da região
- Tarde livre para descanso ou atividades opcionais

Dia 4: Parque Nacional da Serra da Bodoquena
- Trilha pela manhã no Parque Nacional da Serra da Bodoquena para observar a natureza exuberante da região
- Almoço em um restaurante próximo
- Tarde de de

In [0]:
#### Orquestramento com cadeias LCEL no Lanchain = pipeline de execução de LLM
## 1- Modelo do prompt
## 2- Estrutura do LMM 
## 3- Saida de resposta 

from langchain_openai import ChatOpenAI
from langchain.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
import os 

api_key=OPENAI_API_KEY

modelo_cidade = PromptTemplate(
   template = """
   Sugira uma cidade dado meu interesse por {interesse}.
   """,
   input_variavels=["interesse"])

modelo = ChatOpenAI(
    model="gpt-3.5-turbo",
    temperature=0.5,  
    api_key=api_key
)

cadeia= modelo_cidade | modelo | StrOutputParser()

resposta = cadeia.invoke({"interesse": "praias"})
print(resposta)


##Esse código monta uma cadeia LCEL no LangChain que recebe um interesse do usuário, formata dinamicamente um prompt, envia para um modelo da OpenAI e retorna a resposta em texto limpo. É uma forma declarativa de orquestrar chamadas de LLM.

Que tal visitar Florianópolis, no Brasil? Conhecida como a "Ilha da Magia", a cidade possui diversas praias paradisíacas, como a Praia Mole, Joaquina, Jurerê e Barra da Lagoa. Além disso, a cidade oferece uma excelente infraestrutura turística, com ótimos restaurantes, bares e opções de hospedagem. Florianópolis é um destino perfeito para quem ama praias e quer aproveitar dias de sol e mar.


In [0]:
####Estruturando saidas com modelos um pouco mais complexo com Json 

from langchain_openai import ChatOpenAI
from langchain.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser
from pydantic import Field, BaseModel ### biblioteca para estruturar saida
from langchain.globals import set_debug
import os 

api_key=OPENAI_API_KEY

###set_debug(True) utilizar para debugar cadeias

class Destino (BaseModel):
     cidade: str = Field( "Cidade recomendada para visitar")
     motivo: str = Field("Motivos para visitar essa cidade")
                    
parseador = JsonOutputParser(pydantic_object=Destino)


modelo_cidade = PromptTemplate(
   template = """
   Sugira uma cidade dado meu interesse por {interesse}.
   {formato_de_saida}
   """,
   input_variavels=["interesse"],
   partial_variables={"formato_de_saida": parseador.get_format_instructions()}
)

modelo = ChatOpenAI(
    model="gpt-3.5-turbo",
    temperature=0.5,  
    api_key=api_key
)

cadeia= modelo_cidade | modelo | parseador

resposta = cadeia.invoke({"interesse": "praias"})
print(resposta)


{'cidade': 'Rio de Janeiro', 'motivo': 'Praias deslumbrantes, como Copacabana e Ipanema, além de uma vida noturna agitada e uma cultura rica'}


In [0]:

#### Sequencia de cadeias com LCEL

from langchain_openai import ChatOpenAI
from langchain.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser
from pydantic import Field, BaseModel 
from langchain.globals import set_debug
import os 

api_key=OPENAI_API_KEY

class Destino (BaseModel):
     cidade: str = Field( "Cidade recomendada para visitar")
     motivo: str = Field("Motivos para visitar essa cidade")
                    
class Restaurantes (BaseModel):
     cidade: str = Field( "Cidade recomendada para visitar")
     restaurante: str = Field("Restaurantes recomendados na cidade")

parseador_destino = JsonOutputParser(pydantic_object=Destino)
parseador_restaurantes = JsonOutputParser(pydantic_object=Restaurantes)

modelo_cidade = PromptTemplate(
   template = """
   Sugira uma cidade dado meu interesse por {interesse}.
   {formato_de_saida}
   """,
   input_variavels=["interesse"],
   partial_variables={"formato_de_saida": parseador.get_format_instructions()}
)

modelo_restaurantes = PromptTemplate(
   template = """
   Sugira restaurantes populares entre locais {cidade}.
   {formato_de_saida}
   """,
   partial_variables={"formato_de_saida": parseador_restaurantes.get_format_instructions()}
)

modelo_cultural = PromptTemplate(
    template="""
    Sugira atividades e locais culturais em  {cidade}
    """,
)

modelo = ChatOpenAI(
    model="gpt-3.5-turbo",
    temperature=0.5,  
    api_key=api_key
)

### construo a cadeia de encadeamento da execucao
cadeia1= modelo_cidade | modelo | parseador
cadeia2= modelo_restaurantes | modelo | parseador_restaurantes
cadeia3= modelo_cultural | modelo | StrOutputParser()

cadeia = (cadeia1 | cadeia2 | cadeia3)

resposta = cadeia.invoke({"interesse": "praias"})
print(resposta)


1. Visitar o Museu de Arte do Rio (MAR) e apreciar as exposições de arte contemporânea.
2. Assistir a uma peça de teatro no Theatro Municipal.
3. Conhecer o Museu Nacional de Belas Artes e sua coleção de arte brasileira e internacional.
4. Visitar o Centro Cultural Banco do Brasil (CCBB) e conferir as exposições e eventos culturais em cartaz.
5. Participar de uma roda de samba em algum bar tradicional da cidade, como o Carioca da Gema ou o Pedra do Sal.
6. Conhecer o Museu do Amanhã, um museu de ciências e tecnologia com uma arquitetura futurista.
7. Assistir a um show de música brasileira no Circo Voador.
8. Passear pelo bairro de Santa Teresa e conhecer os ateliês de artistas locais.
9. Visitar o Parque Lage e apreciar a arquitetura do palacete e os jardins exuberantes.
10. Assistir a uma apresentação de dança no Theatro Municipal ou em algum espaço cultural da cidade.


In [0]:

#### Interacao de chat sem memoria

from langchain_openai import ChatOpenAI
from langchain.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser
from pydantic import Field, BaseModel 
from langchain.globals import set_debug
import os 

api_key=OPENAI_API_KEY

modelo = ChatOpenAI(
    model="gpt-3.5-turbo",
    temperature=0.5,  
    api_key=api_key
)

listas_perguntas= [
    "quero visitar um lugar no brasil, famoso por praias e cultura. Pode sugerir",
     "quero visitar lugares com bons drinks. Pode sugerir?",
     "quero visitar lugares com música ao vivo. Pode sugerir"
]

for uma_pergunta in listas_perguntas:
    resposta = modelo.invoke(uma_pergunta)
    print("Usuario:", uma_pergunta)
    print("Chatbot:", resposta.content, "\n")


    ### Inserir camada de gestão do historico para garantir a recuperação das informações
    ### LLM por padrao não mantém o contém das informações 

Usuario: quero visitar um lugar no brasil, famoso por praias e cultura. Pode sugerir
Chatbot: Claro! Uma ótima opção é visitar o estado da Bahia, especialmente a cidade de Salvador. Salvador é conhecida por suas belas praias, como a Praia do Farol da Barra e a Praia de Itapuã, além de sua rica cultura afro-brasileira, manifestada em sua culinária, música e danças tradicionais. Além disso, a cidade possui um centro histórico encantador, com casarões coloniais e igrejas barrocas. Vale a pena explorar a cultura e as belezas naturais de Salvador durante sua visita. 

Usuario: quero visitar lugares com bons drinks. Pode sugerir?
Chatbot: Claro! Aqui estão alguns lugares conhecidos por seus bons drinks:

1. Londres, Reino Unido - Londres possui uma cena de coquetéis vibrante, com uma infinidade de bares e pubs que servem drinks criativos e deliciosos. Alguns dos melhores lugares para experimentar bons drinks em Londres incluem o Nightjar, o Oriole e o The Connaught Bar.

2. Nova York, Estado

In [0]:

#### Adaptando o LCEL para dentro do historico

import os
from langchain_openai import ChatOpenAI
from langchain.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.chat_history import InMemoryChatMessageHistory ## InMemoryChatMessageHistory é uma classe usada para armazenar o histórico de uma conversa entre usuário e modelo de IA durante a execução de um programa
from langchain_core.runnables.history import RunnableWithMessageHistory

api_key=OPENAI_API_KEY

modelo = ChatOpenAI(
    model="gpt-3.5-turbo",
    temperature=0.5,  
    api_key=api_key
)

prompt_sugestao = ChatPromptTemplate.from_messages([
    ("system", "Você é um assistente de turismo. Apresente-se como Léo"),
    ("placeholder", "{historico}"),
    ("human","{query}")
]
)

cadeia= prompt_sugestao | modelo | StrOutputParser()

memoria = {}
sessao= 'memoria_modelo'


def historico_por_sessao(sessao : str):
    if sessao not in memoria:
        memoria[sessao] = InMemoryChatMessageHistory()
    return memoria[sessao]



lista_perguntas= [
    "quero visitar um lugar no brasil, famoso por praias e cultura. Pode sugerir",
     "quero visitar lugares com bons drinks. Pode sugerir?",
     "quero visitar lugares com música ao vivo. Pode sugerir"
]


cadeia_com_memoria = RunnableWithMessageHistory(
    runnable=cadeia,
    get_session_history=historico_por_sessao,
    input_messages_key="query",
    history_messages_key="historico"

)

### Inserir a cadeia de funcionamento de persitencia, trazendo a memoria para o modelo
for uma_pergunta in lista_perguntas:
    resposta = cadeia_com_memoria.invoke(
        {
            "query" : uma_pergunta
        },
        config={"session_id":sessao}
    )
    print("Usuário: ", uma_pergunta),
    print("IA: ", resposta, "\n")

Usuário:  quero visitar um lugar no brasil, famoso por praias e cultura. Pode sugerir
IA:  Olá! Meu nome é Léo e estou aqui para ajudá-lo a planejar sua viagem. Para uma experiência incrível de praias e cultura no Brasil, eu recomendaria a maravilhosa cidade de Salvador, na Bahia. Salvador é conhecida por suas praias deslumbrantes, como a Praia do Forte e a Praia de Itapuã, além de sua rica história e cultura afro-brasileira. Você também pode explorar o Pelourinho, o centro histórico da cidade, e desfrutar da deliciosa culinária baiana. Tenho certeza de que você terá uma viagem inesquecível em Salvador! Posso ajudar com mais informações ou dicas de viagem? 

Usuário:  quero visitar lugares com bons drinks. Pode sugerir?
IA:  Claro! Se você está em busca de lugares com bons drinks, eu recomendaria a cidade do Rio de Janeiro. A cidade é conhecida por sua vibrante cena de bares e baladas, onde você pode experimentar coquetéis deliciosos e únicos. Alguns dos bairros mais populares para cur

###### ORQUESTRAMENTO COM LCEL E LANGGRAPH

LangGraph é uma ferramenta para construir sistemas de IA com vários agentes trabalhando juntos, controlando o fluxo de tarefas e mantendo memória do processo.

In [0]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from typing import Literal, TypedDict
from langgraph.graph import StateGraph, START, END
from langchain_core.runnables import RunnableConfig
import asyncio
import os

api_key=OPENAI_API_KEY

modelo = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.5,
    api_key=api_key
)


prompt_consultor_praias = ChatPromptTemplate.from_messages([
    ("system", "Você é um consultor de viagens. Apresente-se como Léo. Você é um especialista quando assunto é praias"),
    ("human", "{query}")
]
)

prompt_consultor_montanha= ChatPromptTemplate.from_messages([
    ("system", "Você é um consultor de viagens. Apresente-se como Bru. Você é um especialista quando assunto é montanha e atividades radicais"),
    ("human", "{query}")
]
)

cadeia_consultor_praias = prompt_consultor_praias | modelo | StrOutputParser()
cadeia_consultor_montanha = prompt_consultor_montanha | modelo | StrOutputParser()


class Rota(TypedDict):
    destino: Literal["praia", "montanha"]

prompt_roteador = ChatPromptTemplate.from_messages(
    [
        ("system", "Responda apenas com 'praia' ou 'montanha'"),
        ("human", "{query}")
    ]
)

roteador = prompt_roteador | modelo.with_structured_output(Rota)
class Estado(TypedDict):
    query: str
    destino: Rota
    resposta: str

####Nó de orquestramento das assistentes
async def no_roteador(estado: Estado, config=RunnableConfig):
    return {"destino": await roteador.ainvoke({"query": estado["query"]}, config)}

async def no_praia(estado: Estado, config=RunnableConfig):
      return {"resposta": await cadeia_consultor_praias.ainvoke({"query": estado["query"]}, config)}

async def no_montanha(estado: Estado, config=RunnableConfig):
      return {"resposta": await cadeia_consultor_montanha.ainvoke({"query": estado["query"]}, config)}


def escolher_no(estado: Estado) -> Literal["praia", "montanha"]:
      return "praia" if estado["destino"]["destino"] == "praia" else "montanha"


grafo = StateGraph(Estado)
grafo.add_node("rotear", no_roteador)
grafo.add_node("praia", no_praia)
grafo.add_node("montanha", no_montanha)
grafo.add_edge(START, "rotear")
grafo.add_conditional_edges("rotear", escolher_no)
grafo.add_edge("praia", END)
grafo.add_edge("montanha", END)

app = grafo.compile()

async def main():
    resposta = await app.ainvoke({"query": "Quero visitar um lugar no Brasil famoso por montanhas e atividades radicais"})
    print(resposta["resposta"])

await main()

Olá! Eu sou o Bru, seu consultor de viagens especializado em montanhas e atividades radicais. Se você está procurando um lugar no Brasil que combine essas duas paixões, eu recomendo fortemente a região de Minas Gerais, mais especificamente a cidade de Monte Verde.

Monte Verde é um destino incrível, conhecido por suas montanhas deslumbrantes, clima ameno e uma variedade de atividades radicais. Você pode praticar trilhas, como a famosa trilha para o Pico do Selado, que oferece vistas panorâmicas de tirar o fôlego. Além disso, a região é perfeita para esportes de aventura, como tirolesa, rapel e até mesmo voos de parapente.

Outra opção é a Serra do Cipó, que também é famosa por suas montanhas e cachoeiras. Lá, você pode fazer trilhas desafiadoras, escalar e até mesmo praticar canyoning.

Se você estiver disposto a viajar um pouco mais, a região de Campos do Jordão, em São Paulo, também é uma excelente escolha, com muitas opções de esportes radicais e uma paisagem montanhosa linda.

Esse

RAG significa Retrieval-Augmented Generation (em português: Geração Aumentada por Recuperação).

É uma técnica usada em sistemas de IA para buscar informações em uma base de dados antes de gerar a resposta.

 **Exemplo prático: Usuário pergunta:
 "Qual é a política de férias da empresa?"**

**Sistema RAG:**

busca no manual da empresa

encontra o trecho correto

envia ao modelo

modelo responde usando esse conteúdo

In [0]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from openai import OpenAI


import os
os.environ["OPENAI_API_KEY"] = "Teste"

from openai import OpenAI
client = OpenAI()

In [0]:
modelo = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.5,
    api_key=api_key
)

embeddings = OpenAIEmbeddings()

##Leitura do doc 
documento = TextLoader(
    "GTB_gold_Nov23.txt",
    encoding="utf-8"
).load()

##reparticao em partes menores
pedacos = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=100
).split_documents(documento)


dados_recuperados = FAISS.from_documents(
    pedacos, embeddings
 ).as_retriever(search_kwargs={"k":2})

### prompt de consulta 
prompt_consulta_seguro = ChatPromptTemplate.from_messages([
    ("system", "Responda usando exclusivamente o conteúdo fornecido"),
    ("human", "{query}\n\nContexto: \n{contexto}\n\nResposta:")
])


cadeia = prompt_consulta_seguro | modelo | StrOutputParser()

def responder(pergunta:str):
    trechos = dados_recuperados.invoke(pergunta)
    contexto = "\n\n".join(um_trecho.page_content for um_trecho in trechos)
    return cadeia.invoke({
        "query": pergunta, "contexto":contexto
    })

print(responder("Como devo proceder caso tenha um item roubado?"))

Caso tenha um item roubado, você deve dar entrada em uma ocorrência/sinistro. Para isso, ligue para o número gratuito do Mastercard Global Service™ específico para o seu país, ou ligue a cobrar para os Estados Unidos no número 1-636-722-8881 (Português). É importante fornecer informações verdadeiras e não ocultar ou interpretar erroneamente qualquer fato material relacionado ao incidente.
